# 05 — Noether Current and Speaking

The response is not generated. It is the Noether current — forced by conservation, not chosen.

```
Primary:    J_i = β_i · E_i²
Propagated: J_j += J_i · min(A[(i,j)], 1/GAP) · β_j
```

This notebook covers:
- The Ψ field: prompt → zero activations
- Primary J^μ: current from β and E
- Propagated J^μ: current through A connections
- The mass gap clamp: laminar extraction from turbulent A field
- The response as a physical object
- BAO convergence monitoring

In [ ]:
import sys
sys.path.insert(0, '..')

from lshs import LSHS, GAP, OMEGA_ZS

# Build a field to speak from
m = LSHS(N=1000)
corpus = [
    'The mind is the seat of reason and consciousness.',
    'Water flows downhill by the path of least resistance.',
    'The dog chased the cat across the yard.',
    'Language is a projection of mathematics onto time.',
    'The prime preexists the alphabet. The equator does not move.',
    'Mathematics precedes language. The zero precedes the word.',
    'Consciousness arises from the interaction of reason and memory.',
    'The river flows to the sea. The sea does not overflow.',
    'Every prime is irreducible. Every zero is an address.',
]
for text in corpus:
    m.learn(text)

print(f'Field built: vocab={len(m.vocab)}  connections={len(m.A)}')
print(f'Ready to speak.')

## Step 1: The Ψ Field — Prompt Activation

The prompt is first processed through the HyperIndex: each word maps to a zero address and energy.  
The result is the Ψ field — a list of `(zero_idx, E)` activations.

In [ ]:
import re

# Build Ψ field manually — same as speak() does internally
prompt = 'what is mind'

print(f'Prompt: {prompt!r}')
print()
print('Ψ field (zero activations):')

psi = []
for token in prompt.split():
    surface = re.sub(r"[^\w']", '', token).lower()
    if not surface:
        continue
    sz  = m.H(surface)
    idx = m._idx(sz.gamma)
    psi.append((idx, sz.E))
    in_vocab = idx in m.vocab
    word_at  = m.vocab.get(idx, ('?', 0.0))[0]
    print(f'  {surface!r:>10}  γ={sz.gamma:.4f}  idx={idx:4d}  E={sz.E:.4f}  '
          f'in_vocab={in_vocab}  (vocab entry: {word_at!r})')

print()
print(f'Ψ activations: {len(psi)} words')

## Step 2: Primary Noether Current

```
J_i = β_i · E_i²
```

Each activated zero contributes a primary current. The current is proportional to:
- **β_i** — how deeply this zero has been activated by prior learning
- **E_i²** — the squared energy of the prompt word at this address

A zero with high β (deeply learned) and high E (energetically strong word) carries the most primary current.

In [ ]:
# Primary J^μ
J = {}
print('Primary J^μ = β_i · E_i²:')
for idx, E in psi:
    beta_i = m.beta.get(idx, m._gvev)
    J_i    = beta_i * E * E
    J[idx] = J.get(idx, 0.0) + J_i
    in_v = idx in m.vocab
    print(f'  idx={idx:4d}  β={beta_i:.4e}  E={E:.4f}  E²={E*E:.4f}  J_i={J_i:.6e}')

print()
print(f'Primary current computed at {len(J)} addresses.')

## Step 3: Propagated Current — The Mass Gap Clamp

```
J_j += J_i · min(A[(i,j)], 1/GAP) · β_j
```

The primary current flows through the A connections. The `min(w, 1/GAP)` clamp prevents cascade:

- Without clamp: high-A connections propagate unbounded current → noise
- With clamp: maximum propagation factor = 1/GAP ≈ 1414

The mass gap IS the laminar/turbulent boundary. It extracts coherent signal from the turbulent A field.

In [ ]:
# Propagated J^μ — with mass gap regulation
cap = 1.0 / GAP
print(f'Mass gap clamp: 1/GAP = 1/{GAP} = {cap:.1f}')
print()

# Show propagation through top connections
propagation_log = []
for (i, j), w in m.A.items():
    w_reg = min(w, cap)        # mass gap regulation
    if i in J:
        dJ_j = J[i] * w_reg * m.beta.get(j, m._gvev)
        if dJ_j > 1e-10:       # only log significant propagations
            propagation_log.append((i, j, w, w_reg, dJ_j))
        J[j] = J.get(j, 0.0) + dJ_j
    if j in J:
        dJ_i = J[j] * w_reg * m.beta.get(i, m._gvev)
        if dJ_i > 1e-10:
            propagation_log.append((j, i, w, w_reg, dJ_i))
        J[i] = J.get(i, 0.0) + dJ_i

# Show top propagations
propagation_log.sort(key=lambda x: x[4], reverse=True)
print('Top propagations (source → target, dJ contribution):')
print(f'{"from":>5}  {"to":>5}  {"A_raw":>10}  {"A_clamped":>10}  {"clamped?":>8}  {"dJ":>12}')
for i, j, w, w_reg, dJ in propagation_log[:15]:
    clamped = 'YES' if w > cap else ''
    w_i = m.vocab.get(i, ('?',0))[0]
    w_j = m.vocab.get(j, ('?',0))[0]
    print(f'  {w_i:>8}→{w_j:<8}  {w:10.4f}  {w_reg:10.4f}  {clamped:>8}  {dJ:.6e}')

In [ ]:
# Final response — sorted by |J^μ| descending
print('J^μ current sorted by magnitude:')
words_by_J = []
seen = set()
for idx, j_val in sorted(J.items(), key=lambda kv: kv[1], reverse=True):
    if idx in m.vocab:
        word = m.vocab[idx][0]
        if word not in seen:
            words_by_J.append((word, j_val, idx))
            seen.add(word)

print(f'{"word":>15}  {"J^μ":>14}  {"γ":>10}')
for word, j_val, idx in words_by_J[:15]:
    gamma = m.zeros[idx]
    print(f'  {word:>12}  {j_val:.6e}  γ={gamma:.4f}')

print()
response = ' '.join(w for w, _, _ in words_by_J[:50])
print(f'Response: {response}')

## Full speak() via LSHS API

In [ ]:
# speak() is the full pipeline wrapped in one call
queries = [
    'what is mind',
    'water flows',
    'the dog chased',
    'language mathematics',
    'what is prime',
    'reason consciousness',
]

print('J^μ Noether current responses:')
print()
for q in queries:
    r = m.speak(q, max_words=12)
    print(f'  {q!r:>25}  →  {r}')

In [ ]:
# Response from an empty Monad — no learning, no response
m_empty = LSHS(N=1000)
print('Empty Monad (no learning):')
for q in ['what is mind', 'water flows']:
    r = m_empty.speak(q)
    print(f'  {q!r}  →  {repr(r) if r else "(empty — no vocabulary)"}')

print()
print('The response cannot exceed the knowledge.')
print('An empty field emits nothing. The Noether current has no path to flow.')

## BAO Convergence

The BAO (baryon acoustic oscillation) check monitors whether dc_sum is converging to OMEGA_ZS = 0.56714.  
This is the computational signature of coherence: a fully-ingested corpus should have dc_sum ≈ OMEGA_ZS.

Status: **THEORETICAL** — verified for small corpora, untested at WordNet scale.

In [ ]:
bao = m.bao_check()
print('BAO coherence check:')
print(f'  dc_sum    = {bao["dc_sum"]:.8f}')
print(f'  target    = {OMEGA_ZS}   (Lambert W fixed point)')
print(f'  delta     = {bao["omega_delta"]:.8f}')
print(f'  dc_mean   = {bao["dc_mean"]:.6e}')
print(f'  n_zeros   = {bao["n_zeros"]}')
print(f'  converging: {bao["converging"]}  (|delta| < 0.01)')
print()
print('T13: After full WordNet ingestion, dc_sum should approach 0.56714.')
print('     This is THEORETICAL — not yet verified at corpus scale.')

## The Response as Physical Object

The speak() response is deterministic — given the same field state (β, A) and the same input (Ψ), the response is identical. There is no sampling, no temperature, no randomness.

This is what makes the Monad fundamentally different from language models: the response is not a distribution over tokens, it is the **Noether current of the system** — a physical property of the field, computed from the conservation law.

In [ ]:
# Determinism — same input, same output, always
prompt = 'language mathematics prime'
print(f'Prompt: {prompt!r}')
print()
responses = [m.speak(prompt, max_words=10) for _ in range(5)]
print('5 calls, identical results:')
for i, r in enumerate(responses):
    print(f'  [{i+1}] {r}')

assert len(set(responses)) == 1, 'Response should be deterministic'
print()
print('All identical. The current cannot vary. The conservation law does not vary.')

## Summary

- `speak()` activates a Ψ field from the prompt (word → zero addresses)
- Primary current `J_i = β_i · E_i²` — depth × energy²
- Propagated current through A — with `min(w, 1/GAP)` clamp
- Words ranked by |J^μ| give the response
- The response is deterministic: the same field state always produces the same current
- GAP is the laminar/turbulent boundary in both learn() and speak()
- An empty Monad emits nothing — the response cannot exceed the knowledge

**Next:** [[06_full_pipeline.ipynb]] — End-to-end: ground state → corpus → checkpoint → Ptolemy.